In [ ]:
### Inspired by https://github.com/eplatero97/coco2mot



import json
from pathlib import Path
import os
from collections import defaultdict
import configparser


opath = "E:\data\\fly_mot_final"
fnames = ["1", "2", "3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20"]  ### 要处理的视频的序号 different video frame folders
coco_json_path = "E:\data\original_frames\\frame_with_full_json"                                         ### video frame folders with corresponding json files

opath = Path(opath)
opath.mkdir(parents = True, exist_ok = True) 
print(opath)
opath.mkdir(parents = True, exist_ok = True) # recursively mkdir if it does not exist

dirs = os.listdir(coco_json_path)
sorted_dirs = sorted(dirs, key=lambda x: int(x))


##### create det/gt/img1 folders for every video sequence
for fname in fnames:
    fpath = opath / fname
    fpath.mkdir(exist_ok = True) # create vid dir
    (fpath / "det").mkdir() # create det dir
    (fpath / "gt").mkdir() # create gt dir
    (fpath / "img1").mkdir() # create img1 dir


for dir in sorted_dirs:
    config = configparser.ConfigParser()
    
    save_path = os.path.join(opath,dir)
    v_path = os.path.join(coco_json_path,dir)                ### all video folders, which includes imgs and corresponding json
    all_obj = defaultdict(list)                              ### all_obj stores all objects in a video 
    for f in os.listdir(v_path):                              ###  go throught all json files
        if f.endswith(".json"):
            with open(os.path.join(v_path,f), 'r') as file:
                js = json.load(file)
            js_shapes = js["shapes"]
            frame_id = int(f.split(".")[0].split("_")[-1]) + 1        #### 如果要从1开始，就要 加 1
            imHeight = js["imageHeight"]
            imWidth = js["imageWidth"]
            for sh in js_shapes:
                if sh["label"].split("-")[0] == "box":
                    tx,ty,w,h = sh["points"][0][0],sh["points"][0][1],sh["points"][1][0]-sh["points"][0][0],sh["points"][1][1]-sh["points"][0][1]
                    # obj_id = sh["label"].split("-")[1]             ###  原始的标注的每个视频中的obj_id 是和视频的序号保持一致的，如果需要修改，则改一下

                    if int(dir) != 10:                               ### object id, if folder name is 10 and object ids in this folder are 001,002, ..., 
                        obj_id = int(sh["label"].split("-")[1]) % (int(dir) *100)
                    else:
                        obj_id = int(sh["label"].split("-")[1])
                    all_obj[obj_id].append([frame_id,tx,ty,w,h,1,1,1]) #### object id in other folders are 100*n + 1, 100*n + 2, ..., for example, obj id in 14 is 1401,1402,...

    # print(all_obj)
    # break
    with open(os.path.join(save_path,"det","det.txt"), "w") as file:
    # 将字符串写入文件
        for key, value in all_obj.items():
            print(key,value)
            for items in value:
                content = f"{items[0]},-1,{items[1]},{items[2]},{items[3]},{items[4]},1,-1,-1,-1\n"
                file.write(content)
    
    with open(os.path.join(save_path,"gt","gt.txt"), "w") as file:
    # 将字符串写入文件
        for key, value in all_obj.items():
            print(key,value)
            for items in value:
                content = f"{items[0]},{key},{items[1]},{items[2]},{items[3]},{items[4]},{items[5]},{items[6]},{items[7]}\n"
                file.write(content)
    
    #### get seqinfo.ini file
    config["Sequence"] = {
    "name": str(dir),
    "imDir": "img1",
    "frameRate": "25",
    "seqLength": "125",
    "imWidth": str(imWidth),
    "imHeight": str(imHeight),
    "imExt": ".jpg"
    }
    with open(os.path.join(save_path,"seqinfo.ini"), "w") as configfile:
        config.write(configfile)

print("Done")

In [ ]:
#### get train.json and val.json for MOT17

### Based ByteTrack/tools/convert_mot17_to_coco.py

import os
import numpy as np
import json
import cv2



DATA_PATH = "E:\data\\fly_mot_final"
OUT_PATH = os.path.join(DATA_PATH, 'annotations')

# SPLITS = ['train_half', 'val_half', 'train', 'test']  # --> split training data to train_half and val_half.

#### Here only have train and val for customed dataset
SPLITS = ['train', 'val']
HALF_VIDEO = False
CREATE_SPLITTED_ANN = False
CREATE_SPLITTED_DET = False


if not os.path.exists(OUT_PATH):
    os.makedirs(OUT_PATH)

for split in SPLITS:
    if split == "val":
        data_path = os.path.join(DATA_PATH, 'val')
    else:
        data_path = os.path.join(DATA_PATH, 'train')
    out_path = os.path.join(OUT_PATH, '{}.json'.format(split))
    out = {'images': [], 'annotations': [], 'videos': [],
            'categories': [{'id': 1, 'name': 'fly'}]}
    seqs = os.listdir(data_path)
    sorted_dirs = sorted(seqs, key=lambda x: int(x))
    image_cnt = 0
    ann_cnt = 0
    video_cnt = 0
    tid_curr = 0
    tid_last = -1
    # print(sorted_dirs)
    # break
    for seq in sorted_dirs:
        video_cnt += 1  # video sequence number.
        # print(seq)
        out['videos'].append({'id': video_cnt, 'file_name': seq})
        seq_path = os.path.join(data_path, seq)
        img_path = os.path.join(seq_path, 'img1')
        ann_path = os.path.join(seq_path, 'gt\\gt.txt')
        images = os.listdir(img_path)
        num_images = len([image for image in images if 'jpg' in image])  # half and half

        if HALF_VIDEO and ('half' in split):
            image_range = [0, num_images // 2] if 'train' in split else \
                        [num_images // 2 + 1, num_images - 1]
        else:
            image_range = [0, num_images - 1]

        for i,img in enumerate(images):
            if i < image_range[0] or i > image_range[1]:
                continue
            # print(i, img)

            # exit()
            # if os.path.isfile(os.path.join(data_path, '{}\\img1\\{:06d}.jpg'.format(seq, i + 1))):
            # f_id = int(img.split('.')[0])
            img = cv2.imread(os.path.join(data_path, '{}/img1/{:06d}.jpg'.format(seq, i + 1)))
            height, width = img.shape[:2]
            image_info = {'file_name': '{}/img1/{:06d}.jpg'.format(seq, i+1),  # image name.
                        'id': image_cnt + i + 1,  # image number in the entire training set.
                        'frame_id': i + 1 - image_range[0],  # image number in the video sequence, starting from 1.
                        'prev_image_id': image_cnt + i if i > 0 else -1,  # image number in the entire training set.
                        'next_image_id': image_cnt + i + 2 if i < num_images - 1 else -1,
                        'video_id': video_cnt,
                        'height': height, 'width': width}
            out['images'].append(image_info)
        # exit()
        print('{}: {} images'.format(seq, num_images))
        if split != 'test':
            det_path = os.path.join(seq_path, 'det\\det.txt')
            anns = np.loadtxt(ann_path, dtype=np.float32, delimiter=',')
            dets = np.loadtxt(det_path, dtype=np.float32, delimiter=',')
            if CREATE_SPLITTED_ANN and ('half' in split):
                anns_out = np.array([anns[i] for i in range(anns.shape[0])
                                    if int(anns[i][0]) - 1 >= image_range[0] and
                                    int(anns[i][0]) - 1 <= image_range[1]], np.float32) 
                anns_out[:, 0] -= image_range[0]
                gt_out = os.path.join(seq_path, 'gt\\gt_{}.txt'.format(split))
                fout = open(gt_out, 'w')
                for o in anns_out:
                    fout.write('{:d},{:d},{:d},{:d},{:d},{:d},{:d},{:d},{:.6f}\n'.format(
                                int(o[0]), int(o[1]), int(o[2]), int(o[3]), int(o[4]), int(o[5]),
                                int(o[6]), int(o[7]), o[8]))
                fout.close()
            if CREATE_SPLITTED_DET and ('half' in split):
                dets_out = np.array([dets[i] for i in range(dets.shape[0])
                                    if int(dets[i][0]) - 1 >= image_range[0] and
                                    int(dets[i][0]) - 1 <= image_range[1]], np.float32)
                dets_out[:, 0] -= image_range[0]
                det_out = os.path.join(seq_path, 'det\\det_{}.txt'.format(split))
                dout = open(det_out, 'w')
                for o in dets_out:
                    dout.write('{:d},{:d},{:.1f},{:.1f},{:.1f},{:.1f},{:.6f}\n'.format(
                                int(o[0]), int(o[1]), float(o[2]), float(o[3]), float(o[4]), float(o[5]),
                                float(o[6])))
                dout.close()

            print('{} ann images'.format(int(anns[:, 0].max())))
            for i in range(anns.shape[0]):
                frame_id = int(anns[i][0])
                if frame_id - 1 < image_range[0] or frame_id - 1 > image_range[1]:
                    continue
                track_id = int(anns[i][1])
                cat_id = int(anns[i][7])
                ann_cnt += 1
                if not ('15' in DATA_PATH):
                    #if not (float(anns[i][8]) >= 0.25):  # visibility.
                        #continue
                    if not (int(anns[i][6]) == 1):  # whether ignore.
                        continue
                    if int(anns[i][7]) in [3, 4, 5, 6, 9, 10, 11]:  # Non-person
                        continue
                    if int(anns[i][7]) in [2, 7, 8, 12]:  # Ignored person
                        category_id = -1
                    else:
                        category_id = 1  # pedestrian(non-static)
                        if not track_id == tid_last:
                            tid_curr += 1
                            tid_last = track_id
                else:
                    category_id = 1
                ann = {'id': ann_cnt,
                    'category_id': category_id,
                    'image_id': image_cnt + frame_id,
                    'track_id': tid_curr,
                    'bbox': anns[i][2:6].tolist(),
                    'conf': float(anns[i][6]),
                    'iscrowd': 0,
                    'area': float(anns[i][4] * anns[i][5])}
                out['annotations'].append(ann)
        image_cnt += num_images
        print(tid_curr, tid_last)
    print('loaded {} for {} images and {} samples'.format(split, len(out['images']), len(out['annotations'])))
    json.dump(out, open(out_path, 'w'), indent=4)